# Week 3 – Feast Feature Store for the IRIS pipeline
Roll no. 21f2000602 · Offline store: local Parquet (Task 6: BigQuery) · Online store: SQLite

In [1]:
%pip install "feast[gcp]" scikit-learn pandas joblib dvc dvc-gs

INFO: pip is looking at multiple versions of gcsfs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of gcsfs to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of fsspec[tqdm] to determine which version is compatible with other requirements. This could take a while.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of pyopenssl to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versi

 - Checking Feast Version

In [1]:
!feast version

Feast SDK Version: "0.67.0"


### Initializing Feast

In [2]:
!feast init iris_feature_store


Creating a new Feast repository in /home/jupyter/21f2000602_MLOPS_WEEKLY_ASSIGNMENT/iris_feature_store.



In [3]:
!find iris_feature_store -type f -not -path "*/__pycache__/*"

iris_feature_store/README.md
iris_feature_store/feature_repo/feature_store.yaml
iris_feature_store/feature_repo/feature_definitions.py
iris_feature_store/feature_repo/test_workflow.py
iris_feature_store/feature_repo/__init__.py
iris_feature_store/feature_repo/data/driver_stats.parquet
iris_feature_store/feature_repo/data/driver_quality_labels.parquet
iris_feature_store/__init__.py
iris_feature_store/.gitignore


 - Checking Config file

In [4]:
!cat iris_feature_store/feature_repo/feature_store.yaml

project: iris_feature_store
# By default, the registry is a file (but can be turned into a more scalable SQL-backed registry)
registry: data/registry.db
# The provider primarily specifies default offline / online stores & storing the registry in a given cloud
provider: local
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 3
# By default, no_auth for authentication and authorization, other possible values kubernetes and oidc. Refer the documentation for more details.
auth:
    type: no_auth


- Explicitly write the oofline store config for later use

In [5]:
%%writefile -a iris_feature_store/feature_repo/feature_store.yaml
offline_store:
    type: file                   # offline (historical) store = local Parquet files via pandas

Appending to iris_feature_store/feature_repo/feature_store.yaml


In [6]:
!cat iris_feature_store/feature_repo/feature_store.yaml

project: iris_feature_store
# By default, the registry is a file (but can be turned into a more scalable SQL-backed registry)
registry: data/registry.db
# The provider primarily specifies default offline / online stores & storing the registry in a given cloud
provider: local
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 3
# By default, no_auth for authentication and authorization, other possible values kubernetes and oidc. Refer the documentation for more details.
auth:
    type: no_auth
offline_store:
    type: file                   # offline (historical) store = local Parquet files via pandas


#### (Optional) Removing Unnecessary files

In [7]:
!cd iris_feature_store/feature_repo && rm -f feature_definitions.py example_repo.py test_workflow.py data/driver_*.parquet
!rm -f iris_feature_store/README.md
!find iris_feature_store -type f -not -path "*/__pycache__/*"

iris_feature_store/feature_repo/feature_store.yaml
iris_feature_store/feature_repo/__init__.py
iris_feature_store/__init__.py
iris_feature_store/.gitignore


#### Keep Generated files out of git

In [8]:
%%writefile iris_feature_store/feature_repo/.gitignore
# Generated by Feast - rebuilt by `feast apply` / `feast materialize`, never committed
data/registry.db
data/online_store.db
# Parquet copy of the dataset - regenerated by the notebook from the DVC-tracked CSV
data/*.parquet

Writing iris_feature_store/feature_repo/.gitignore


### Downloading and Versioning Data

In [9]:
!dvc import https://github.com/IITMBSMLOps/ga_resources.git iris_data_adapted_for_feast.csv \
    -o iris_feature_store/feature_repo/data/iris_data_adapted_for_feast.csv --rev week_3

Importing 'iris_data_adapted_for_feast.csv (https://github.com/IITMBSMLOps/ga_resources.git)' -> 'iris_feature_store/feature_repo/data/iris_data_adapted_for_feast.csv'
                                                                               
To track the changes with git, run:

	git add iris_feature_store/feature_repo/data/iris_data_adapted_for_feast.csv.dvc iris_feature_store/feature_repo/data/.gitignore

To enable auto staging, run:

	dvc config core.autostage true


- Data Exploration

In [10]:
import pandas as pd

csv_path = "iris_feature_store/feature_repo/data/iris_data_adapted_for_feast.csv"
raw = pd.read_csv(csv_path, parse_dates=["event_timestamp", "created_timestamp"])

print(raw.shape)                                     # rows, columns
print(raw.groupby("iris_id")["species"].agg(["count", "first"]))   # rows + species per plant
print(raw["event_timestamp"].min(), "→", raw["event_timestamp"].max())
raw.head()

(45, 8)
         count       first
iris_id                   
1001        15  versicolor
1002        15      setosa
1003        15      setosa
2025-09-17 10:40:17.102131 → 2025-10-01 10:40:17.102131


,event_timestamp,iris_id,sepal_length,sepal_width,petal_length,petal_width,species,created_timestamp
0,2025-09-17 10:40:17.102131,1001,5.52,2.53,3.86,1.13,versicolor,2025-10-02 10:40:17.172178
1,2025-09-18 10:40:17.102131,1001,5.50,2.24,3.60,1.08,versicolor,2025-10-02 10:40:17.172178
2,2025-09-19 10:40:17.102131,1001,5.55,2.47,3.75,1.08,versicolor,2025-10-02 10:40:17.172178
3,2025-09-20 10:40:17.102131,1001,5.45,2.37,3.92,1.20,versicolor,2025-10-02 10:40:17.172178
4,2025-09-21 10:40:17.102131,1001,5.65,2.52,3.95,1.17,versicolor,2025-10-02 10:40:17.172178


- Convert to parquet

In [11]:
for col in ["event_timestamp", "created_timestamp"]:
    raw[col] = raw[col].dt.tz_localize("UTC")        # mark times as UTC

parquet_path = "iris_feature_store/feature_repo/data/iris_data.parquet"
raw.to_parquet(parquet_path, index=False)
print(pd.read_parquet(parquet_path).dtypes)

event_timestamp      datetime64[ns, UTC]
iris_id                            int64
sepal_length                     float64
sepal_width                      float64
petal_length                     float64
petal_width                      float64
species                           object
created_timestamp    datetime64[ns, UTC]
dtype: object


### Writing Features for the data

In [12]:
%%writefile iris_feature_store/feature_repo/iris_features.py
from datetime import timedelta

from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float64
from feast.value_type import ValueType

# ENTITY: the "thing" features describe. join_keys = the column that identifies it.
iris = Entity(
    name="iris",
    join_keys=["iris_id"],
    value_type=ValueType.INT64,
    description="One iris plant, tracked over time",
)

# DATA SOURCE: where raw feature values live + which column says WHEN each was measured.
iris_source = FileSource(
    name="iris_source",
    path="data/iris_data.parquet",                 # relative to this feature repo
    timestamp_field="event_timestamp",             # when the measurement happened
    created_timestamp_column="created_timestamp",  # when the row was written (tie-breaker)
)

# FEATURE VIEW: which columns are features, for which entity, from which source.
iris_features = FeatureView(
    name="iris_features",
    entities=[iris],
    ttl=timedelta(days=365 * 5),                   # how old a value may be and still count
    schema=[
        Field(name="sepal_length", dtype=Float64),
        Field(name="sepal_width",  dtype=Float64),
        Field(name="petal_length", dtype=Float64),
        Field(name="petal_width",  dtype=Float64),
    ],
    online=True,                                   # allow materializing into the online store
    source=iris_source,
)

Writing iris_feature_store/feature_repo/iris_features.py


- CHecking File write

In [13]:
!cat iris_feature_store/feature_repo/iris_features.py | head -15
!ls -la iris_feature_store/feature_repo/data

from datetime import timedelta

from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float64
from feast.value_type import ValueType

# ENTITY: the "thing" features describe. join_keys = the column that identifies it.
iris = Entity(
    name="iris",
    join_keys=["iris_id"],
    value_type=ValueType.INT64,
    description="One iris plant, tracked over time",
)

# DATA SOURCE: where raw feature values live + which column says WHEN each was measured.
total 28
drwxr-xr-x 2 jupyter jupyter 4096 Oct 25 17:19 .
drwxr-xr-x 4 jupyter jupyter 4096 Oct 25 17:22 ..
-rw-r--r-- 1 jupyter jupyter   33 Oct 25 17:18 .gitignore
-rw-r--r-- 1 jupyter jupyter 6670 Oct 25 17:19 iris_data.parquet
-rw-r--r-- 1 jupyter jupyter 4009 Oct 25 17:18 iris_data_adapted_for_feast.csv
-rw-r--r-- 1 jupyter jupyter  344 Oct 25 17:18 iris_data_adapted_for_feast.csv.dvc


### Apply definitions & materialize into the online store

In [14]:
%%bash
cd iris_feature_store/feature_repo
feast apply

No project found in the repository. Using project name iris_feature_store defined in feature_store.yaml
Applying changes for project iris_feature_store
Created project iris_feature_store
Created entity iris
Created feature view iris_features

Created sqlite table iris_feature_store_iris_features



- Verify Changes

In [15]:
%%bash
cd iris_feature_store/feature_repo
feast entities list
feast feature-views list

NAME    DESCRIPTION                        TYPE
iris    One iris plant, tracked over time  ValueType.INT64
NAME           ENTITIES    TYPE         ENABLED    STATE
iris_features  {'iris'}    FeatureView  Yes        STATE_UNSPECIFIED


- Check by looking inside the store

In [17]:
import sqlite3

db = "iris_feature_store/feature_repo/data/online_store.db"
table = "iris_feature_store_iris_features"

con = sqlite3.connect(db)
print("Tables:", con.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall())
print("Rows before materialize:", con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0])
con.close()

Tables: [('iris_feature_store_iris_features',)]
Rows before materialize: 0


#### Materialize

In [20]:
%%bash
cd iris_feature_store/feature_repo
feast materialize 2025-09-01T00:00:00 2025-10-02T00:00:00

Materializing 1 feature views from 2025-09-01 00:00:00+00:00 to 2025-10-02 00:00:00+00:00 into the sqlite online store.

iris_features:


- verify Materialization

In [21]:
from datetime import datetime, timezone

con = sqlite3.connect(db)
n = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
print("Rows after materialize:", n)

for feat, ts in con.execute(f"SELECT feature_name, event_ts FROM {table} LIMIT 4"):
    print(f"{feat:13s} event_ts = {datetime.fromtimestamp(ts, tz=timezone.utc)}")
con.close()

Rows after materialize: 12
sepal_length  event_ts = 2025-10-01 10:40:17+00:00
sepal_width   event_ts = 2025-10-01 10:40:17+00:00
petal_length  event_ts = 2025-10-01 10:40:17+00:00
petal_width   event_ts = 2025-10-01 10:40:17+00:00


In [22]:
!cd iris_feature_store/feature_repo && feast feature-views list

NAME           ENTITIES    TYPE         ENABLED    STATE
iris_features  {'iris'}    FeatureView  Yes        AVAILABLE_ONLINE
